# Python ThreadPool
Each program is a process and has at least one thread that executes instructions for that process. The underlying operating system controls how new threads are created, when threads are executed, and which CPU core executes them. Sometimes we may need to create and start new threads to run additional tasks concurrently.

In [8]:
from threading import Thread
from multiprocessing.pool import ThreadPool
from time import sleep
import os
from random import random
# Project
from socket import AF_INET, SOCK_STREAM, socket

In [6]:
def task():
    print("This is another thread")
    
# Whenever we create new threads, it is good practice to protect the entry point of the program. 
if __name__ == '__main__':
    # define a task to run in a new thread
    thread = Thread(target=task)
    # start the task in a new thread. The main thread then blocks until the new thread terminates.
    thread.start()
    # wait for the thread to terminate
    thread.join()

This is another thread


A **thread pool** is a programming pattern for automatically managing pool of worker threads:

* It controls when they are created, such as when they are needed.
* It controls how many tasks each worker can execute before being replaced.
* It also controls what workers should do when they are not being used, such as making them wait without consuming computational resources.

In [10]:
def task():
    print("This is another thread")
    
if __name__ == '__main__':
    # context manager: create and configure the thread pool.
    # It ensures the pool is closed once we are finished using it.
    with ThreadPool() as pool:
        # exectue a function asynchronously (non-blocking)
        async_result = pool.apply_async(task)
        # wait for an asynchonous task to complete
        async_result.wait()

This is another thread


**When to use the ThreadPool?**
* Your tasks can be defined by a pure function that has no state or side effects.
* Your task can fit within a single Python function, likely making it simple and easy to understand.
* You need to perform the same task many times with different arguments, e.g., homogenous tasks.
* You need to apply the same function to each object in a collection in a for-loop.
* IO-bound tasks which involve reading from or writing to a device, file, or socket connection. Speed is bound by the device, hard drive, or network connection.

**Configure the ThreadPool via the Constructor**
Things we can configure:
* `processes`: Maximum number of worker threads (not processes) to use in the pool.
* `initializer`: Function executed after each worker thread is created.
* `initargs`: Arguments to the worker thread initialization function.

Default Number Worker Threads = Total Logical CPU Cores in Your System

In [17]:
print("Number of logical cores: " + str(os.cpu_count()))

# Check for hyperthreading

Number of logical cores: 8


In [16]:
# protect the entry point
if __name__ == '__main__':
    # create a thread pool
    pool = ThreadPool()
    # report the status of the thread pool
    print(pool)
    # close the thread pool
    pool.close()

<multiprocessing.pool.ThreadPool state=RUN pool_size=8>


In [ ]:
# create a thread pool with 4 workers
pool = ThreadPool(processes=4)

You'll likely have many more thread workers than you have physical or logical CPU cores. This is because threads are lightweight units of concurrency and we often require and easily support tens, hundreds, or even thousands of concurrent threads on modern systems. Experiment, start with 100.

In [21]:
# protect the entry point
if __name__ == '__main__':
    # create a thread pool
    pool = ThreadPool(100)
    # report the status of the thread pool
    print(pool)
    # close the thread pool
    pool.close()

<multiprocessing.pool.ThreadPool state=RUN pool_size=100>


In [24]:
def task ():
    print("Worker executing task...")
    # block for a moment
    sleep(1)
    
def init():
    print("Initializing worker...")
    
if __name__ == "__main__":
    with ThreadPool(processes=2, initializer=init) as pool:
        # issue tasks to the thread pool
        for _ in range(4):
            pool.apply_async(task)
        pool.close()
        # wait for all tasks to complete
        pool.join()

Initializing worker...Initializing worker...

Worker executing task...Worker executing task...

Worker executing task...Worker executing task...



## Synchronously (blocking)
* `apply()`: For executing one-off-tasks.
* `map()`: For calling a function many times with different arguments.
* `starmap()`: For calling a function many times with multiple different arguments.

In [27]:
def task():
    print("This is another thread")

# pool.apply(task, args=(arg1, arg2, arg3)) -> if you want to pass arguments to your function
if __name__ == '__main__':
    with ThreadPool() as pool:
        # issue a task and wait for it to complete
        pool.apply(task)

This is another thread


`items` is an iterable. A `chunksize` argument can be specified to split the tasks into groups which may be sent to each worker thread to be executed in batch.
```python
for result in pool.map(task, items, chunksize=10):
    print(result)
```

In [29]:
def task(arg):
    print(f"From another thread {arg}")
    return arg * 2

if __name__ == "__main__":
    with ThreadPool() as pool:
        # Issue multiple tasks and handle return values
        for result in pool.map(task, range(10)):
            print(result)

From another thread 0From another thread 1

From another thread 2
From another thread 3
From another thread 4
From another thread 5
From another thread 6
From another thread 7
From another thread 8
From another thread 9
0
2
4
6
8
10
12
14
16
18


In [32]:
def task(arg1, arg2, arg3):
    print(f"From another thread {arg1}, {arg2}, {arg3}")
    return arg1 + arg2 + arg3

if __name__ == "__main__":
    with ThreadPool() as pool:
        # prepare task arguments: [(0, 0, 0), (1, 2, 3), (2, 4, 6), ..., (9, 18, 27)]
        args = [(i, i*2, i*3) for i in range(10)]
        # issue multiple tasks and handle return values
        for result in pool.starmap(task, args):
            print(result)

From another thread 0, 0, 0From another thread 1, 2, 3

From another thread 2, 4, 6
From another thread 3, 6, 9
From another thread 4, 8, 12
From another thread 5, 10, 15
From another thread 6, 12, 18
From another thread 7, 14, 21
From another thread 8, 16, 24
From another thread 9, 18, 27
0
6
12
18
24
30
36
42
48
54


## Asynchronously (non-blocking)
* `apply_async()`: For executing one-off tasks.
* `map_async()`: For calling a function many times with different arguments. 
* `starmap_async()`: For calling a function many times with multiple different arguments.

In [33]:
def task():
    print("This is another thread")

# pool.apply_async(task, args=(arg1, arg2)) -> if you want to pass arguments to your function 
if __name__ == "__main__":
    with ThreadPool() as pool:
        # issue a task asynchronously
        async_result = pool.apply_async(task) 
        # wait for the task to complete
        async_result.wait()

This is another thread


In [34]:
def task(arg):
    print(f"From another thread {arg}")
    return arg * 2

if __name__ == "__main__":
    with ThreadPool() as pool:
        # Issue multiple tasks to the pool
        async_result = pool.map_async(task, range(10))
        # Handle return values once all tasks completed
        for result in async_result.get():
            print(result)

From another thread 0From another thread 1
From another thread 2

From another thread 3
From another thread 4From another thread 5
From another thread 6
From another thread 7
From another thread 8From another thread 9


0
2
4
6
8
10
12
14
16
18


In [35]:
def task(arg1, arg2, arg3):
    print(f"From another thread {arg1}, {arg2}, {arg3}")
    return arg1 + arg2 + arg3

if __name__ == "__main__":
    with ThreadPool() as pool:
        # prepare task arguments: [(0, 0, 0), (1, 2, 3), (2, 4, 6), ..., (9, 18, 27)]
        args = [(i, i*2, i*3) for i in range(10)]
        # issue multiple tasks to the pool
        async_result = pool.starmap_async(task, args)
        # handle return values
        for result in async_result.get():
            print(result)

From another thread 0, 0, 0From another thread 1, 2, 3
From another thread 2, 4, 6
From another thread 3, 6, 9
From another thread 4, 8, 12
From another thread 5, 10, 15
From another thread 6, 12, 18
From another thread 7, 14, 21
From another thread 8, 16, 24
From another thread 9, 18, 27

0
6
12
18
24
30
36
42
48
54


## Limitations of `map()`
A problem with the `map()` method on the `ThreadPool` is that traverses the provided iterable and issues all tasks on the `ThreadPool` immediately.

This can be a problem if the iterable contains many hundreds or thousands of items. This is because the `ThreadPool` will then have hundreds, thousands, or millions of tasks sitting idly waiting to execute, using large amounts of main memory unnecessarily.

Solution: `imap()`:
* Argument items are yielded from the iterable as workers become available, rather than all at once.
* Return values are yielded in order as they are completed, rather than after all tasks are completed.

A limitation of `imap()` method is that it yields return value results in the order that the tasks were issued to the `Threadpool`. Although results are yielded as tasks are completed, the caller may not be as responsive as it could be if later tasks complete before earlier tasks, holding up the progress of the iterable of return values.

The `imap_unordered()` addresses this limitation. It returns an iterable of return values. The return values are yielded in the order the tasks are completed, not the order that the tasks were issued to the `ThreadPool`.

In [39]:
# Custom function to be executed in a worker thread
def task(arg):
    # block for a random fraction of a second
    sleep(random())
    print(f"From another thread {arg}")
    return arg * 2

if __name__ == "__main__":
    with ThreadPool(4) as pool:
        # issue multiple tasks and handle return values
        for result in pool.imap(task, range(10)):
            print(result)   
            
        
# This is different from the map() method that would issue all tasks at once, then wait for all
# issued tasks to complete returning an iterable of return values.

From another thread 2
From another thread 3
From another thread 1
From another thread 4
From another thread 0
0
2
4
6
8
From another thread 7
From another thread 9
From another thread 5
10
From another thread 8
From another thread 6
12
14
16
18


In [40]:
def task(arg):
    # block for a random fraction of a second
    sleep(random())
    print(f"From another thread {arg}")
    return arg * 2

if __name__ == "__main__":
    with ThreadPool(4) as pool:
        # issue multiple tasks and handle return values
        for rs in pool.imap_unordered(task, range(10)):
            print(rs)

From another thread 1
2
From another thread 0
0
From another thread 2
4
From another thread 4
8
From another thread 3
6
From another thread 6
12
From another thread 9
18
From another thread 7
14
From another thread 5
10
From another thread 8
16


## Callbacks
A callback is a function that is first registered and then called automatically by the `ThreadPool` on some event. They are called in 2 situations:
* **Result**: With the results of a task when the task finished successfully.
```python
result = apply_async(..., callback=result_callback)
```
* **Error**: When an exception or error is raised in a task and is not handled.
```python
result = apply_async(..., error_callback=callback=result_callback)
```

It can be used with the following methods: `apply_sync()`, `map_async()`, `starmap_async()`





In [5]:
def result_callback(return_value):
    print(f"Callback got: {return_value}")
    
def task(identifier):
    """
    Generates a random number, reports the number, blocks for a moment, then returns the value that was generated.
    """
    value = random()
    print(f"Task {identifier} executing with {value}")
    sleep(value)
    return value

if __name__ == '__main__':
    with ThreadPool() as pool:
        # issue tasks to the thread pool
        result = pool.apply_async(task, args=(0, ), callback=result_callback)
        # close the thread pool
        pool.close()
        # wait for all tasks to complete
        pool.join()

Task 0 executing with 0.06146570949475105
Callback got: 0.06146570949475105


## Managing Asynchronous Tasks & Checking Status of Tasks with `AsyncResult`
1. Retrieving results:
```python
value = async_result.get()
```
---
```python
try:
    value = async_result.get(timeout=10)
except TimeoutError as e:
    ...
```
2. Wait for all tasks to finish.
```python
# wait for issued task to complete
async_result.wait()
```
---
    A timeout argument can be specified to set a limit in seconds for how long the caller is willing to wait. We can check if the tasks completed via the `ready()` method.
```python
# wait for issued task to complete with a timeout
async_result.wait(timeout=10)
# check if the tasks are all done
if async_result.ready()
    print('All Done')
    ...
else :
    print('Not Done Yet')
    ...
```
3. Check if tasks are completed.
```python
# check if tasks are still running
if async_result.ready():
    print('Tasks are done')
else:
    print('Tasks are not done')
```
4. Check if tasks finished normally. If at least one issued task raised an exception, then the call was not successful and the `successful()` method will return False.
```python
# check if the tasks have completed
if async_result.ready():
    # check if the tasks were successful
    if async_result.successful():
        print('Successful')
    else:
        print('Unsuccessful')
```
    If the issue tasks are still running, a ValueError is raised and may need to be handled
```python
try:
    # check if the tasks were successful
    if async_result.successful():
        print('Successful')
except ValueError as e:
    print('Tasks still running')
```

**Example**

In [7]:
def task():
    """
    Long running task to run asynchronously. Task will loop 10 times and each iteration it will generate a random
    number between 0 and 1, block for a fraction of a second then report the value that was generated.
    """
    for i in range(10):
        value = random()
        sleep(value)
        print(f">{i} got {value}")

if __name__ == '__main__':
    with ThreadPool() as pool:
        # issue a task asynchronously
        async_result = pool.apply_async(task)
        # wait until the task is complete
        while not async_result.ready():
            print("Main thread waiting...")
            # block for a moment
            async_result.wait(timeout=1)
        # report if the task was successful
        if async_result.successful():
            print("Task was successful.")

Main thread waiting...
>0 got 0.4365008594950466
>1 got 0.4690452345924935
Main thread waiting...
>2 got 0.8213491132311829
Main thread waiting...
>3 got 0.9933746170676966
Main thread waiting...
>4 got 0.5690076859962878
>5 got 0.13439554266758125
>6 got 0.06794088505691598
Main thread waiting...
>7 got 0.5800800383740681
>8 got 0.39370961366077073
Main thread waiting...
>9 got 0.653012406538605
Task was successful.


## Case Study: Developing a Port Scanner
We can connect to other computers by opening a socket, called socket programming. Opening a socket requires both the name or IP address of the server and a port number on which to connect. For example, when your browser opens a web page on python.org, it is opening a socket connection to that server on port 80, then using the HTTP protocol to request and download (`GET`) an HTML file.

Port scanner is a program that reports all of the open sockets on a given server. A simply way to implement a port scanner is to loop over all the ports you want to test and attempt to make a socket connection on each. If a connection can be made, we disconnect immediately and report that the port on the server is open.

Python provides a socket communication in the socket module.

We will attempt to open TCP sockets in this case, as they are more commonly used for services like SMTP (email), HTTP (web), FTP (files), and so on.

**Slow**
```python
# We can configure our socket for TCP using the SOCK_STREAM constant
sock = socket(AF_INET, SOCK_STREAM)
```

In [9]:
def test_port_number(host, port):
    # Create and configure the socket
    with socket(AF_INET, SOCK_STREAM) as sock:
        # It is a good idea to set a timeout because attempting to open network connections can be slow.
        # We want to give up connecting and raise an exception if a given number of seconds elapses
        # and we still haven’t connected
        sock.settimeout(3)
        # connecting may fail
        try:
            # Attempt to connect. Requires a host name and a port.
            sock.connect((host, port))
            # a successful connection was made
            return True
        except:
            # ignore the failure
            return False

# scan port numbers on a host
def port_scan(host, ports):
    print(f"Scanning {host}...")
    # scan each port number
    for port in ports:
        if test_port_number(host, port):
            print(f"> {host}:{port} open")
            
# protect the entry point
if __name__ == '__main__':
    # define host and port numbers to scan
    host = "python.org"
    # Many common internet services are provided on port numbers between 0 and 1,024, and as
    # such we will limit our scanning to this range. The viable range of port numbers is 0 to 65,535,
    ports = range(1024)
    # test the ports
    port_scan(host, ports)

Scanning python.org...
> python.org:80 open
> python.org:443 open


**Fast**

In [11]:
# returns True if can connect, False otherwise
def test_port_number(host, port):
    # create and configure the socket
    with socket(AF_INET, SOCK_STREAM) as sock:
        # set a timeout of a few seconds
        sock.settimeout(3)
        # connecting might fail
        try:
            # attempt to connect
            sock.connect((host, port))
            # a successful connection was made
            return True
        except:
            # ignore the failure
            return False
        
# scan port numbers on a host
def port_scan(host, ports):
    print(f"Scanning {host}...")
    # create the thread pool
    with ThreadPool(len(ports)) as pool:
        # prepare arguments for starmap
        args = [(host, p) for p in ports]
        # dispatch all tasks
        results = pool.starmap(test_port_number, args)
        # report results and port numbers together
        for port, is_open in zip(ports, results):
            if is_open:
                print(f"> {host}:{port} open")
                
# protect the entry point
if __name__ == "__main__":
    # define host and port numbers to scan
    host = "python.org"
    ports = range(1024)
    # test the ports
    port_scan(host, ports)
    
# 71 faster

Scanning python.org...
> python.org:80 open
> python.org:443 open


$ \text{Speedup} = \frac{\text{slow_time}}{\text{fast_time}} $